# Jarvis: Çok Modlu Sesli Yapay Zekâ Asistanı
Bu Colab defteri, sesli sohbet edebilen, bilgi tabanı oluşturabilen ve kullanıcı etkileşimlerinden öğrenebilen çok modlu bir yapay zekâ asistanı sağlar. Her hücreyi sırayla çalıştırarak eksiksiz bir kurulumu tamamlayabilirsiniz.

## 1. Ortamı Hazırlama
Aşağıdaki adımlar GPU durumunu kontrol eder ve gereken tüm Python/OS paketlerini kurar. Colab'de çalıştırdığınızdan emin olun; ilk kurulum birkaç dakika sürebilir.

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Kullanılacak cihaz: {device}')
if device == 'cpu':
    print('GPU bulunamadı. Colab Runtime > Change runtime type menüsünden GPU seçebilirsiniz.')

In [ ]:
# Gerekli Python ve sistem paketlerini kurun
!apt-get install -qq libespeak1 > /dev/null
!pip install -q accelerate==0.28.0 bitsandbytes==0.43.1 datasets faiss-cpu gradio==4.21.0 "sentence-transformers>=2.5.0"     soundfile==0.12.1 TTS==0.19.1 torch torchvision torchaudio transformers==4.39.3 wikipedia     openai-whisper==20231117

## 2. Kütüphaneleri İçe Aktarma ve Genel Ayarlar
Model bileşenleri, konuşma tanıma ve konuşma sentezleyici bu bölümde hazırlanır.

In [ ]:
import gc
import json
import os
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import faiss
import gradio as gr
import numpy as np
import soundfile as sf
import wikipedia
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    GenerationConfig
)
from TTS.api import TTS
import whisper

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

## 3. Bilgi Üssü (Retrieval) ve Bellek Altyapısı
Kullanıcının sağladığı dokümanları veya internetten gelen bilgileri (örneğin Wikipedia) vektör uzayında saklayıp arayabilmek için aşağıdaki yardımcı sınıflar tanımlanmıştır.

In [ ]:
@dataclass
class KnowledgeDocument:
    text: str
    metadata: Dict[str, str] = field(default_factory=dict)

class KnowledgeBase:
    def __init__(self, model_name: str = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'):
        self.embedder = SentenceTransformer(model_name)
        self.dim = self.embedder.get_sentence_embedding_dimension()
        self.index = faiss.IndexFlatIP(self.dim)
        self.documents: List[KnowledgeDocument] = []
        self.normalized = True
        print(f'Bilgi tabanı hazır: {model_name}')

    def _embed(self, texts: List[str]) -> np.ndarray:
        embeddings = self.embedder.encode(texts, convert_to_numpy=True, normalize_embeddings=self.normalized)
        return embeddings.astype('float32')

    def add_documents(self, docs: List[KnowledgeDocument]):
        if not docs:
            return
        texts = [doc.text for doc in docs]
        vectors = self._embed(texts)
        self.index.add(vectors)
        self.documents.extend(docs)
        print(f'{len(docs)} doküman eklendi. Toplam: {len(self.documents)}')

    def search(self, query: str, top_k: int = 3) -> List[Tuple[KnowledgeDocument, float]]:
        if not self.documents:
            return []
        query_vec = self._embed([query])
        scores, indices = self.index.search(query_vec, top_k)
        results = []
        for idx, score in zip(indices[0], scores[0]):
            if idx == -1:
                continue
            results.append((self.documents[idx], float(score)))
        return results

    def to_json(self, path: str):
        payload = [{'text': doc.text, 'metadata': doc.metadata} for doc in self.documents]
        Path(path).write_text(json.dumps(payload, ensure_ascii=False, indent=2))
        print(f'Bilgi tabanı {path} dosyasına kaydedildi.')

    def load_json(self, path: str):
        if not Path(path).exists():
            raise FileNotFoundError(path)
        payload = json.loads(Path(path).read_text())
        docs = [KnowledgeDocument(**item) for item in payload]
        self.index.reset()
        self.documents = []
        self.add_documents(docs)
        print(f'{path} dosyasından {len(docs)} kayıt yüklendi.')

In [ ]:
class ConversationMemory:
    def __init__(self, max_turns: int = 12):
        self.max_turns = max_turns
        self.turns: List[Tuple[str, str]] = []

    def add_turn(self, user: str, assistant: str):
        self.turns.append((user, assistant))
        if len(self.turns) > self.max_turns:
            self.turns = self.turns[-self.max_turns:]

    def to_text(self) -> str:
        formatted = []
        for idx, (u, a) in enumerate(self.turns, start=1):
            formatted.append(f"Tur {idx}:
Kullanıcı: {u}
Asistan: {a}")
        return '

'.join(formatted) if formatted else 'Henüz kayıt yok.'

## 4. Konuşma Tanıma (STT) ve Konuşma Sentezleyici (TTS)
Whisper tabanlı konuşma tanıma ve çok dilli Coqui TTS modeliyle ses çıktısı üretimi hazırlanır.

In [ ]:
class SpeechRecognizer:
    def __init__(self, model_size: str = 'small'):
        print('Whisper modeli yükleniyor...')
        self.model = whisper.load_model(model_size, device=device)

    def transcribe(self, audio_path: str, language: Optional[str] = None) -> str:
        result = self.model.transcribe(audio_path, language=language)
        return result.get('text', '').strip()

class TextToSpeechSynthesizer:
    def __init__(self, model_name: str = 'tts_models/tr/common-voice/glow-tts', vocoder: str = 'vocoder_models/universal/libri-tts/fullband-melgan'):
        print('TTS modeli yükleniyor...')
        self.tts = TTS(model_name=model_name, progress_bar=False, gpu=device == 'cuda')
        self.vocoder_name = vocoder

    def synthesize(self, text: str, speaker: Optional[str] = None) -> Tuple[int, np.ndarray]:
        wav = self.tts.tts(text=text, speaker=speaker, vocoder_name=self.vocoder_name)
        wav = np.array(wav, dtype=np.float32)
        sample_rate = self.tts.synthesizer.output_sample_rate
        return sample_rate, wav

    def save(self, text: str, path: str) -> str:
        sr, wav = self.synthesize(text)
        sf.write(path, wav, sr)
        return path

## 5. Dil Modeli ve Yardımcı Asistan Sınıfı
TinyLlama tabanlı dil modeli, bilgi tabanı ve konuşma geçmişi ile birlikte yanıtlar üretir.

In [ ]:
class ConversationalAgent:
    def __init__(self, model_name: str = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0', max_new_tokens: int = 256):
        print('Dil modeli yükleniyor...')
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
            low_cpu_mem_usage=True,
            device_map='auto' if device == 'cuda' else None
        )
        self.generation_config = GenerationConfig(
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9
        )
        self.memory = ConversationMemory()
        self.knowledge_base = KnowledgeBase()
        self.system_prompt = (
            'Sen Jarvis adında çok yetenekli, güvenilir ve açıklayıcı bir yapay zekâ asistanısın. '
            'Kullanıcının taleplerine adım adım, gerçekçi ve güvenli çözümler sun. '
            'Bilgiyi bilmediğinde tahminde bulunma, bunun yerine nasıl öğrenebileceğini açıkla. '
            'Türkçe veya İngilizce yanıt verebilirsin.'
        )

    def build_prompt(self, user_message: str, retrieved: List[Tuple[KnowledgeDocument, float]]) -> str:
        context_lines = []
        for doc, score in retrieved:
            meta = ', '.join(f"{k}: {v}" for k, v in doc.metadata.items()) if doc.metadata else ''
            context_lines.append(f'- (skor={score:.2f}) {meta}
{doc.text}')
        context_block = '
'.join(context_lines) if context_lines else 'İlgili bilgi bulunamadı.'
        memory_block = self.memory.to_text()
        prompt = f"<|system|>{self.system_prompt}</s><|memory|>{memory_block}</s><|context|>{context_block}</s><|user|>{user_message}</s><|assistant|>"
        return prompt

    def chat(self, user_message: str, knowledge_results: Optional[List[Tuple[KnowledgeDocument, float]]] = None) -> Dict[str, str]:
        knowledge_results = knowledge_results or []
        prompt = self.build_prompt(user_message, knowledge_results)
        inputs = self.tokenizer(prompt, return_tensors='pt').to(self.model.device)
        with torch.no_grad():
            output_ids = self.model.generate(**inputs, generation_config=self.generation_config)
        generated_text = self.tokenizer.decode(output_ids[0], skip_special_tokens=True)
        if '<|assistant|>' in generated_text:
            assistant_reply = generated_text.split('<|assistant|>')[-1].strip()
        else:
            assistant_reply = generated_text.strip()
        self.memory.add_turn(user_message, assistant_reply)
        return {
            'response': assistant_reply,
            'knowledge_used': [doc.text for doc, _ in knowledge_results],
            'prompt_tokens': inputs['input_ids'].shape[-1]
        }

    def add_external_knowledge(self, text: str, metadata: Optional[Dict[str, str]] = None):
        doc = KnowledgeDocument(text=text, metadata=metadata or {})
        self.knowledge_base.add_documents([doc])

    def search_knowledge(self, query: str, top_k: int = 3):
        return self.knowledge_base.search(query, top_k=top_k)

    def wikipedia_lookup(self, query: str, sentences: int = 3):
        try:
            summary = wikipedia.summary(query, sentences=sentences, auto_suggest=True)
            self.add_external_knowledge(summary, metadata={'kaynak': 'Wikipedia', 'baslik': query})
            return summary
        except Exception as exc:
            return f'Wikipedia sorgusu başarısız oldu: {exc}'

    def reset_memory(self):
        self.memory = ConversationMemory()
        print('Konuşma geçmişi sıfırlandı.')

## 6. Modelleri Başlatma
Aşağıdaki hücre, dil modeli, konuşma tanıma ve konuşma sentezi bileşenlerini hazırlar. Çalıştırma süresi cihazınıza göre değişebilir.

In [ ]:
agent = ConversationalAgent()
speech_to_text = SpeechRecognizer(model_size='small')
text_to_speech = TextToSpeechSynthesizer()

## 7. Bilgi Edinme ve Öğrenme Akışı Örneği
Aşağıdaki yardımcı fonksiyon, kullanıcı isteğine göre bilgi tabanını zenginleştirir ve yanıt oluşturur.

In [ ]:
def enrich_and_respond(query: str, use_wikipedia: bool = True) -> Dict[str, str]:
    retrieved = agent.search_knowledge(query)
    if not retrieved and use_wikipedia:
        summary = agent.wikipedia_lookup(query)
        if not summary.startswith('Wikipedia sorgusu başarısız'):
            retrieved = agent.search_knowledge(query)
    result = agent.chat(query, knowledge_results=retrieved)
    result['retrieved_docs'] = [doc.text for doc, _ in retrieved]
    return result

# Örnek kullanım
demo_result = enrich_and_respond('Yapay zekâ nedir?')
print(json.dumps(demo_result, ensure_ascii=False, indent=2))

## 8. Sesli ve Yazılı Arayüz (Gradio)
Bu arayüz sayesinde mikrofondan konuşabilir veya metin girişi yapabilirsiniz. Yanıt hem metin hem de sentezlenmiş ses olarak dönecektir.

In [ ]:
def interact(audio_input, text_input, language, use_wiki):
    user_text = text_input.strip() if text_input else ''
    transcription = ''
    if audio_input is not None:
        temp_audio_path = 'temp_input.wav'
        sr, audio = audio_input
        sf.write(temp_audio_path, audio, sr)
        transcription = speech_to_text.transcribe(temp_audio_path, language=language or None)
        user_text = transcription

    if not user_text:
        return '', 'Herhangi bir metin alınamadı.', None

    result = enrich_and_respond(user_text, use_wikipedia=use_wiki)
    response_text = result['response']
    sr, wav = text_to_speech.synthesize(response_text)
    return response_text, transcription or user_text, (sr, wav)

with gr.Blocks() as app:
    gr.Markdown('# Jarvis Sesli Asistan')
    gr.Markdown('Metin girin veya mikrofondan konuşun. Yanıt, bilgi tabanından destek alır.')
    with gr.Row():
        audio_in = gr.Audio(sources=['microphone', 'upload'], type='numpy', label='Ses Girişi')
        text_in = gr.Textbox(label='Metin Girişi', placeholder='Sorunuzu yazın veya ses kaydedin...')
    with gr.Row():
        language = gr.Dropdown(['tr', 'en', ''], value='tr', label='Whisper Dil Seçimi (boş bırakırsanız otomatik)')
        use_wiki = gr.Checkbox(value=True, label='Wikipedia ile otomatik bilgi edin')
    send_button = gr.Button('Jarvis ile Konuş')
    with gr.Row():
        text_out = gr.Textbox(label='Jarvis Yanıtı')
        transcript_out = gr.Textbox(label='Algılanan Kullanıcı Metni')
    audio_out = gr.Audio(label='Sesli Yanıt', type='numpy')

    send_button.click(
        interact,
        inputs=[audio_in, text_in, language, use_wiki],
        outputs=[text_out, transcript_out, audio_out]
    )

app

## 9. Arayüzü Başlatma
Aşağıdaki hücreyi çalıştırarak Gradio uygulamasını başlatın. Colab ortamında otomatik olarak paylaşılabilir bir bağlantı üretilecektir.

In [ ]:
app.launch(share=True)

## 10. İsteğe Bağlı: Bilgi Tabanını Dışa Aktarma
Güncellenen bilgi tabanını JSON olarak kaydedebilir ve daha sonra tekrar yükleyebilirsiniz.

In [ ]:
# Örnek kullanım
knowledge_path = 'jarvis_knowledge.json'
agent.knowledge_base.to_json(knowledge_path)
agent.knowledge_base.load_json(knowledge_path)